# 🚨 Adaptive Incident Choreographer — Colab Training Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/COolAlien35/AIC/blob/main/train_colab.ipynb)

This notebook demonstrates how to train the AIC orchestrator agent using
**HuggingFace TRL's PPOTrainer** with LoRA-based parameter-efficient fine-tuning.

**What this notebook does:**
1. Installs all dependencies (openenv, trl, transformers, peft, etc.)
2. Loads the AIC OpenEnv-compliant environment
3. Runs a minimal PPO training loop on the environment
4. Logs reward curves and saves a checkpoint

## 1. Install Dependencies

In [ ]:
# Clone the AIC repo
!git clone https://github.com/COolAlien35/AIC.git
%cd AIC

# Install core dependencies
!pip install -q openenv==0.1.13
!pip install -q torch==2.2.2
!pip install -q transformers==4.40.2
!pip install -q trl==0.8.6
!pip install -q peft==0.11.1
!pip install -q pydantic==2.7.1
!pip install -q numpy==1.26.4
!pip install -q pandas==2.2.2
!pip install -q rich==13.7.1
!pip install -q accelerate>=0.28.0
!pip install -q datasets
!pip install -e .

print("All dependencies installed.")

## 2. Verify OpenEnv Environment

In [ ]:
from openenv.env import Env as OpenEnvBase
from aic.env.aic_environment import AICEnvironment

# Verify AICEnvironment inherits from OpenEnvBase
assert issubclass(AICEnvironment, OpenEnvBase), "AICEnvironment must inherit from OpenEnv!"
print(f"✅ AICEnvironment inherits from: {OpenEnvBase.__module__}.{OpenEnvBase.__name__}")
print(f"   MRO: {[c.__name__ for c in AICEnvironment.__mro__]}")

# Quick smoke test
env = AICEnvironment(episode_id=0, base_seed=42)
obs = env.reset()
print(f"\n✅ Environment reset. Observation keys: {list(obs.keys())}")
print(f"   SLA remaining: {obs['sla_remaining_steps']} steps")
print(f"   State space: {env.state_space}")
print(f"   Action space: {env.action_space}")
print(f"   Max episode length: {env.episode_max_length}")

## 3. Run a Baseline Episode (Rule-Based)

In [ ]:
from aic.training.config import TrainingConfig
from aic.training.train import run_episode
from aic.utils.seeding import make_episode_rng, get_adversary_cycle
from aic.agents.db_agent import DBAgent
from aic.agents.infra_agent import InfraAgent
from aic.agents.app_agent import AppAgent
from aic.agents.adversarial_agent import AdversarialAgent
from aic.agents.orchestrator_agent import OrchestratorAgent

# Rule-based baseline episode
config = TrainingConfig(num_episodes=1, use_llm_agents=False)
db = DBAgent(use_llm=False)
infra = InfraAgent(use_llm=False)
app = AppAgent(use_llm=False)
cycle = get_adversary_cycle(make_episode_rng(0, 42))
adv = AdversarialAgent(cycle, db)
orch = OrchestratorAgent(adv, use_llm=False)

result = run_episode(0, config, orch, db, infra, app)
print(f"Baseline episode reward: {result['total_reward']:+.2f}")
print(f"Final health: {result['final_health']:.3f}")

## 4. Setup PPO Training with TRL

We use a small language model (`Qwen/Qwen2.5-3B-Instruct`) with LoRA
to keep GPU memory manageable on Colab. The model learns to generate
incident-response actions that maximize AIC's composite reward.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead

# ── Model & Tokenizer ───────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── LoRA config ──────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

# ── PPO config ───────────────────────────────────────────────────────
ppo_config = PPOConfig(
    model_name=MODEL_NAME,
    learning_rate=1e-4,
    batch_size=4,
    mini_batch_size=2,
    ppo_epochs=2,
    gradient_accumulation_steps=2,
    log_with=None,  # Set to "wandb" if you want W&B logging
)

# ── Load model with value head ──────────────────────────────────────
model = AutoModelForCausalLMWithValueHead.from_pretrained(
    MODEL_NAME,
    peft_config=lora_config,
    torch_dtype=torch.float32,
    device_map={"": device},
)

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=model,
    tokenizer=tokenizer,
)

print(f"\n✅ PPOTrainer initialized with {MODEL_NAME}")
print(f"   LoRA rank: {lora_config.r}, alpha: {lora_config.lora_alpha}")
print(f"   Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 5. PPO Training Loop over AIC Environment

Each training iteration:
1. Build a prompt from the AIC environment observation
2. Generate an action with the LM
3. Step the environment and get a reward
4. Update the policy with PPO

In [ ]:
from aic.env.aic_environment import AICEnvironment
from aic.env.reward_engine import compute_r1
from aic.utils.constants import SLA_STEPS
import json

NUM_EPISODES = 3        # Keep small for Colab demo
STEPS_PER_EPISODE = 20  # SLA_STEPS

episode_rewards = []

for episode_id in range(NUM_EPISODES):
    env = AICEnvironment(episode_id=episode_id, base_seed=42)
    obs = env.reset()
    episode_reward = 0.0

    # Collect (query, response, reward) tuples for PPO batch
    queries = []
    responses = []
    rewards_list = []

    for step_idx in range(STEPS_PER_EPISODE):
        # ── Build prompt from observation ────────────────────────────
        prompt = (
            f"You are an SRE orchestrator. Step {obs['step']}/{SLA_STEPS}.\n"
            f"SLA remaining: {obs['sla_remaining_steps']} steps.\n"
            f"Alerts:\n{obs['alert_summary_text'][:500]}\n\n"
            f"Output a concrete remediation action in JSON:\n"
        )

        # Tokenize the prompt
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

        # ── Generate action with the LM ─────────────────────────────
        with torch.no_grad():
            gen_ids = model.generate(
                input_ids,
                max_new_tokens=128,
                temperature=0.3,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
            )
        response_ids = gen_ids[0][input_ids.shape[1]:]
        action_text = tokenizer.decode(response_ids, skip_special_tokens=True)

        # ── Step environment ────────────────────────────────────────
        obs, reward, done, info = env.step(action_text)

        # Use R1 (health recovery) as the step reward signal
        metrics = env.world_state.snapshot()
        r1_reward = compute_r1(metrics)
        step_reward = r1_reward

        # Accumulate for PPO
        queries.append(input_ids.squeeze(0))
        responses.append(response_ids)
        rewards_list.append(torch.tensor(step_reward, dtype=torch.float32, device=device))
        episode_reward += step_reward

        if done:
            break

    # ── PPO update ──────────────────────────────────────────────────
    # Batch the collected data and run PPO step
    if len(queries) >= ppo_config.batch_size:
        batch_queries = queries[:ppo_config.batch_size]
        batch_responses = responses[:ppo_config.batch_size]
        batch_rewards = rewards_list[:ppo_config.batch_size]

        stats = ppo_trainer.step(batch_queries, batch_responses, batch_rewards)
        print(
            f"Episode {episode_id}: reward={episode_reward:+.2f}, "
            f"health={info.get('health', 0):.3f}, "
            f"ppo/loss={stats.get('ppo/loss/total', 'N/A')}"
        )
    else:
        print(
            f"Episode {episode_id}: reward={episode_reward:+.2f}, "
            f"health={info.get('health', 0):.3f} (batch too small for PPO)"
        )

    episode_rewards.append(episode_reward)

print(f"\n{'='*60}")
print(f"Training complete. {NUM_EPISODES} episodes.")
print(f"Avg reward: {sum(episode_rewards)/len(episode_rewards):+.2f}")

## 6. Save Checkpoint

In [ ]:
from pathlib import Path

ckpt_dir = Path("checkpoints/ppo_colab")
ckpt_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(ckpt_dir))
tokenizer.save_pretrained(str(ckpt_dir))

print(f"✅ Checkpoint saved to {ckpt_dir}")
print(f"   Files: {[f.name for f in ckpt_dir.iterdir()]}")

## 7. Reward Curve Visualization

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(range(len(episode_rewards)), episode_rewards, marker='o', linewidth=2, color='#3b82f6')
plt.fill_between(range(len(episode_rewards)), episode_rewards, alpha=0.1, color='#3b82f6')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('AIC PPO Training — Reward Curve')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Summary

This notebook demonstrated:
- ✅ **OpenEnv compliance** — `AICEnvironment` inherits from `openenv.env.Env`
- ✅ **PPO training** — via HuggingFace TRL's `PPOTrainer` with LoRA
- ✅ **Reward integration** — R1 health-recovery reward drives policy updates
- ✅ **Checkpoint saving** — model weights + tokenizer saved for reuse

For full 100-episode training, increase `NUM_EPISODES` and use a GPU runtime.

**Links:**
- [GitHub Repository](https://github.com/COolAlien35/AIC)
- [Live Demo on HF Spaces](https://huggingface.co/spaces/COolAlien35/AIC)
- [Blog Post](https://huggingface.co/blog/COolAlien35/adaptive-incident-choreographer)